In [1]:
import pandas as pd
import datetime as dt
from datetime import datetime

import math
import os
#https://www.programiz.com/python-programming/datetime/strftime


In [2]:

start_date = dt.date(2019, 1, 1)
end_date = dt.date(2030, 12, 31)

# 1. Creación base eficiente
Calendar = pd.DataFrame({'Fecha': pd.date_range(start_date, end_date)})

# --- FECHA ---
Calendar['Date'] = Calendar['Fecha'].dt.strftime('%m-%d-%Y')

# --- DAY ---
Calendar['Day Week'] = (Calendar['Fecha'].dt.dayofweek + 1).map(lambda x: f"{x:02d}")
Calendar['Day Month'] = Calendar['Fecha'].dt.day.map(lambda x: f"{x:02d}")

# --- WEEK ---
Calendar['Week of Month'] = Calendar['Fecha'].dt.day.apply(lambda x: math.ceil(x/7)).map(lambda x: f"{x:02d}")
Calendar['Week of Year'] = (Calendar['Fecha'].dt.strftime('%U').astype(int) + 1).map(lambda x: f"{x:02d}")

# --- ISO (Aquí corregimos el error del 29-Dic-2025) ---
iso_df = Calendar['Fecha'].dt.isocalendar() # Esto obtiene Year, Week y Day ISO correctamente
Calendar['Week of Year ISO'] = iso_df['week'].map(lambda x: f"{x:02d}")
Calendar['Year_ISO'] = iso_df['year'].astype(str)
Calendar['Year-Week ISO'] = Calendar['Year_ISO'] + Calendar['Week of Year ISO']

# Week ID ISO (Corregido para ser numérico incremental)
Calendar['Week ID ISO'] = ((Calendar['Fecha'] - Calendar['Fecha'].min()).dt.days // 7 + 1).map(lambda x: f"{x:02d}")

# --- MONTH ---
Calendar['Month of Year'] = Calendar['Fecha'].dt.month.map(lambda x: f"{x:02d}")
Calendar['Month name'] = Calendar['Fecha'].dt.month_name()

# --- QUARTER ---
Calendar['Quarter No'] = Calendar['Fecha'].dt.quarter.map(lambda x: f"{x:02d}")
Calendar['Quarter'] = "Q" + Calendar['Quarter No']

# --- YEAR ---
Calendar['Year'] = Calendar['Fecha'].dt.year.astype(int)

# --- YEAR -> MONTH - WEEK (Standard) ---
Calendar['Year-Month'] = Calendar['Year'].astype(str) + Calendar['Month of Year']
Calendar['Year-Week'] = Calendar['Year'].astype(str) + Calendar['Week of Year']

# --- YEAR-MONTH ISO (Lógica corregida) ---
# Si la semana ISO pertenece al año siguiente, el mes ISO debe alinearse con ese año
def get_year_month_iso(row):
    # Si la fecha es de diciembre pero la semana ISO ya es del año siguiente
    if row['Fecha'].month == 12 and int(row['Week of Year ISO']) < 5:
        return f"{int(row['Year'])+1}01"
    # Si la fecha es de enero pero la semana ISO es del año anterior (ej. semana 52 o 53)
    elif row['Fecha'].month == 1 and int(row['Week of Year ISO']) > 50:
        return f"{int(row['Year'])-1}12"
    else:
        return f"{row['Year']}{row['Month of Year']}"

Calendar['Year-Month ISO'] = Calendar.apply(get_year_month_iso, axis=1)

# --- GUARDAR ARCHIVOS ---
# Nota: Asegúrate de que las carpetas existan o usa os.makedirs
# ruta_drive = r'...' 
ruta_drive = r'C:\Users\SSN0609\OneDrive - Stanley Black & Decker\Latin America - Regional Marketing - Marketing Analytics\Data\Processed-Dataflow\Shared_Information_for_Projects\Calendar\Calendar.csv'

Calendar.to_csv(ruta_drive, index=False)

